### Polymarket Calibration Engine

This notebook analyzes the forecasting accuracy of Polymarket by computing Brier Scores and calibration curves for resolved markets using the Dune API. It reproduces the core charts from my Dune dashboard and makes the analysis fully portable outside the Dune UI.

**What this notebook does**

- Queries Dune for ~100k+ resolved Polymarket markets via `dune-client`[web:119]  
- Computes how average Brier Score evolves from 30 days before resolution down to 1 day before  
- Breaks Brier Score decay out by market category (Politics, Crypto, Sports, Finance/Macro, Esports/Gaming, Other)  
- Builds a calibration curve 7 days before resolution to test whether implied probabilities match actual outcomes  

Brier Score is a standard forecast accuracy metric defined as \( (p - o)^2 \), where \(p\) is the predicted probability and \(o \in \{0,1\}\) is the realized outcome. Lower scores indicate better calibration.

In [1]:
!pip install dune-client -q

from dune_client.client import DuneClient
from dune_client.query import QueryBase

API_KEY = "YOUR API KEY"  # paste your key from dune.com/settings/api

client = DuneClient(api_key=API_KEY)

# Test with the counting query (fastest, single number)
query = QueryBase(query_id=7393683)
results = client.run_query_dataframe(query)
print(results)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   total_resolved_markets
0                  894945


In [2]:
import pandas as pd

# Brier Score Decay - All Markets
query_brier = QueryBase(query_id=7392114)
df_brier = client.run_query_dataframe(query_brier)
print("Brier Decay shape:", df_brier.shape)
print(df_brier.head(3))

Brier Decay shape: (30, 4)
   days_before_resolution  n_markets  avg_brier_score  avg_winning_token_price
0                      30      22077         0.072509                 0.857602
1                      29      22896         0.072738                 0.857211
2                      28      23734         0.072487                 0.857302


In [3]:
# Brier Score Decay by Category
query_brier_cat = QueryBase(query_id=7392485)
df_brier_cat = client.run_query_dataframe(query_brier_cat)
print("Brier Decay by Category shape:", df_brier_cat.shape)
print(df_brier_cat.head(3))

# Calibration Curve (7 days)
query_calib = QueryBase(query_id=7393378)
df_calib = client.run_query_dataframe(query_calib)
print("Calibration shape:", df_calib.shape)
print(df_calib.head(3))

Brier Decay by Category shape: (180, 4)
   days_before_resolution        category  n_markets  avg_brier_score
0                      30          Crypto       1821         0.087749
1                      30  Esports/Gaming         45         0.117757
2                      30   Finance/Macro       1736         0.088318
Calibration shape: (10, 5)
   price_bucket  midpoint  perfect_calibration  n_tokens  actual_win_rate
0           0.0      0.05                 0.05     13264         0.098537
1           0.1      0.15                 0.15     14989         0.164921
2           0.2      0.25                 0.25     12433         0.254564


In [4]:
import plotly.express as px

# Make sure days are sorted from 30 → 1
df_brier_sorted = df_brier.sort_values("days_before_resolution", ascending=False)

fig1 = px.line(
    df_brier_sorted,
    x="days_before_resolution",
    y="avg_brier_score",
    title="Polymarket Brier Score Decay (All Markets)",
    labels={
        "days_before_resolution": "Days Before Resolution",
        "avg_brier_score": "Average Brier Score"
    }
)

fig1.update_traces(mode="lines+markers")
fig1.update_layout(
    xaxis=dict(autorange="reversed")  # 30 on the left, 1 on the right
)
fig1.show()

## Brier Score Decay

The first chart shows how Polymarket's average Brier Score changes as markets approach resolution.

- **X‑axis:** Days before resolution (30 on the left → 1 on the right)  
- **Y‑axis:** Average Brier Score of winning tokens at that horizon  

We expect Brier Scores to **decrease** closer to resolution as information arrives and prices move toward 0 or 1. The second chart repeats the same analysis but splits markets by **category** to see which types of markets calibrate fastest (e.g., Politics vs Sports vs Esports).

In [5]:
# Brier Score Decay by Category

# Ensure proper ordering of days and categories
df_brier_cat_sorted = df_brier_cat.sort_values(
    ["days_before_resolution", "category"],
    ascending=[False, True]
)

fig2 = px.line(
    df_brier_cat_sorted,
    x="days_before_resolution",
    y="avg_brier_score",
    color="category",
    title="Polymarket Brier Score Decay by Category",
    labels={
        "days_before_resolution": "Days Before Resolution",
        "avg_brier_score": "Average Brier Score",
        "category": "Category"
    }
)

fig2.update_traces(mode="lines+markers")
fig2.update_layout(
    xaxis=dict(autorange="reversed")  # 30 on the left, 1 on the right
)
fig2.show()

## Calibration Curve (7 Days Before Resolution)

The calibration curve asks a stricter question:

> When Polymarket prices an outcome at X% seven days before resolution, does it actually occur X% of the time?

Construction:

- For each resolved market token, take its **average price 6–8 days before resolution** as the implied probability  
- Bucket tokens into 10‑point probability bands (0–10%, 10–20%, …, 90–100%)  
- For each bucket, compute the **actual win rate** of those tokens  

In the scatter plot:

- Each point is a bucket (e.g., 0.25 on the x‑axis is the 20–30% bucket)  
- The orange dashed line is **perfect calibration** (implied probability = actual win rate)  
- The blue points show Polymarket's actual performance

Points close to the diagonal indicate good calibration. Points **below** the diagonal indicate overconfidence (implied probability too high relative to realized frequency), especially in the highest probability bucket.

In [6]:
# Calibration Curve (7 days before resolution)

fig3 = px.scatter(
    df_calib,
    x="midpoint",
    y="actual_win_rate",
    title="Polymarket Calibration Curve (7 Days Before Resolution)",
    labels={
        "midpoint": "Implied Probability (7 Days Out)",
        "actual_win_rate": "Actual Win Rate"
    }
)

# Add perfect calibration line
fig3.add_scatter(
    x=df_calib["midpoint"],
    y=df_calib["perfect_calibration"],
    mode="lines",
    name="Perfect Calibration",
    line=dict(color="orange", dash="dash")
)

fig3.update_traces(marker=dict(size=8))
fig3.update_layout(
    yaxis=dict(range=[0, 1]),
    xaxis=dict(range=[0, 1])
)
fig3.show()